# Punkt 2. Ocena zależności występowania braków danych od pozostałych cech

## Cel analizy

Celem analizy jest sprawdzenie, czy prawdopodobieństwo braku wartości zmiennej `y_t2` jest powiązane z wartościami pozostałych cech zbioru danych. Wyniki pozwolą wskazać zmienne, które mogą być pomocne przy imputacji brakujących wartości `y_t2`.

W analizie wykorzystano wskaźnik braków danych oraz dwie metody oceny zależności:
1. porównanie średnich wartości cech ciągłych w grupie obserwacji z brakującą i zaobserwowaną wartością `y_t2`;
2. ocenę zależności wskaźnika braków od cech skokowych za pomocą tabel kontyngencji i miar asocjacji.

Dodatkowo zastosowano regresję logistyczną, aby ocenić jednoczesny związek wszystkich pozostałych cech z występowaniem braku `y_t2`.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"
CSV_FILE = DATA / "ka01.csv"

df = pd.read_csv(CSV_FILE)

print(f"Liczba obserwacji: {len(df)}")
print(f"Liczba zmiennych: {df.shape[1]}")
display(df.head())

Liczba obserwacji: 805
Liczba zmiennych: 5


,y_t1,x1,trt,x2,y_t2
0,9.632983,55.158388,1,0,7.764858
1,11.885582,49.207020,1,1,10.669919
2,10.449324,47.741301,1,1,11.215358
3,9.399263,39.062814,1,0,NaN
4,11.186338,49.632284,1,0,9.510397


## 1. Przygotowanie wskaźnika braków danych

W zbiorze braki występują w zmiennej `y_t2`. Dla tej zmiennej utworzono wskaźnik `R_y_t2`, który przyjmuje wartość 1, gdy obserwacja jest brakująca, oraz 0, gdy wartość została zaobserwowana.

W dalszej analizie zmienną `R_y_t2` traktuje się jako zmienną zależną, a pozostałe cechy jako potencjalne zmienne wyjaśniające występowanie braków.

In [2]:
df["R_y_t2"] = df["y_t2"].isna().astype(int)

predictors = ["y_t1", "x1", "trt", "x2"]

if df[predictors].isna().any().any():
    raise ValueError("W zmiennych objaśniających występują braki danych.")

n_missing = int(df["R_y_t2"].sum())
n_observed = int((df["R_y_t2"] == 0).sum())

missing_summary = pd.DataFrame({
    "Liczba obserwacji": [n_observed, n_missing],
    "Udział (%)": [
        100 * n_observed / len(df),
        100 * n_missing / len(df)
    ]
}, index=["y_t2 zaobserwowane (R=0)", "y_t2 brakujące (R=1)"])

display(missing_summary.round(2))

,Liczba obserwacji,Udział (%)
y_t2 zaobserwowane (R=0),544,67.58
y_t2 brakujące (R=1),261,32.42


## 2. Porównanie średnich cech ciągłych

Pierwsza metoda polega na podziale obserwacji na dwie grupy: z brakującą i zaobserwowaną wartością `y_t2`. Następnie porównywane są średnie wartości cech ciągłych w tych grupach.

Dla każdej cechy zastosowano test t Welcha. Test ten pozwala porównać średnie bez przyjmowania założenia o równości wariancji w obu grupach.

Hipotezy testowe:

- \(H_0\): średnie wartości analizowanej cechy w obu grupach są równe;
- \(H_1\): średnie wartości analizowanej cechy w obu grupach różnią się.

Dodatkowo wyznaczono różnicę średnich oraz jej 95-procentowy przedział ufności. Wartości p skorygowano metodą Holma ze względu na jednoczesne testowanie dwóch cech.

In [3]:
continuous_results = []

for variable in ["y_t1", "x1"]:
    group_missing = df.loc[df["R_y_t2"] == 1, variable]
    group_observed = df.loc[df["R_y_t2"] == 0, variable]

    test = stats.ttest_ind(
        group_missing,
        group_observed,
        equal_var=False
    )

    difference = group_missing.mean() - group_observed.mean()

    # Przedział ufności dla różnicy średnich w teście Welcha
    se = np.sqrt(
        group_missing.var(ddof=1) / len(group_missing)
        + group_observed.var(ddof=1) / len(group_observed)
    )
    df_welch = test.df
    critical_value = stats.t.ppf(0.975, df_welch)
    ci_low = difference - critical_value * se
    ci_high = difference + critical_value * se

    continuous_results.append({
        "Cecha": variable,
        "Średnia R=1": group_missing.mean(),
        "Średnia R=0": group_observed.mean(),
        "Różnica średnich (R=1 minus R=0)": difference,
        "95% CI – dolna granica": ci_low,
        "95% CI – górna granica": ci_high,
        "Statystyka t": test.statistic,
        "Stopnie swobody": df_welch,
        "p-value": test.pvalue
    })

continuous_results = pd.DataFrame(continuous_results)
continuous_results["p-value skorygowane (Holm)"] = multipletests(
    continuous_results["p-value"],
    method="holm"
)[1]

display(continuous_results.round(4))

,Cecha,Średnia R=1,Średnia R=0,Różnica średnich (R=1 minus R=0),95% CI – dolna granica,95% CI – górna granica,Statystyka t,Stopnie swobody,p-value,p-value skorygowane (Holm)
0,y_t1,11.0189,10.8223,0.1967,0.0506,0.3427,2.6457,511.8729,0.0084,0.0168
1,x1,52.1112,52.5905,-0.4793,-1.3750,0.4164,-1.0512,527.4022,0.2936,0.2936


### Interpretacja wyników

Wyniki testów wskazują, że średnia wartość `y_t1` różni się między grupą z brakującym i zaobserwowanym `y_t2`. Różnica jest dodatnia, co oznacza, że średnia `y_t1` jest wyższa w grupie, w której brakuje `y_t2`.

W przypadku `x1` nie stwierdzono istotnej statystycznie różnicy średnich. Oznacza to, że w badanej próbie nie ma podstaw do stwierdzenia związku tej cechy z występowaniem braków `y_t2` na podstawie zastosowanego testu.

## 3. Zależność wskaźnika braków od cech skokowych

Drugą metodę zastosowano do zmiennych skokowych `trt` i `x2`. Dla każdej z nich utworzono tabelę kontyngencji, przedstawiającą liczebność obserwacji z brakującą i zaobserwowaną wartością `y_t2`.

Zależność między cechą skokową a wskaźnikiem braków oceniono testem chi-kwadrat niezależności. Siłę związku określono za pomocą współczynnika phi.

Hipotezy testowe:

- \(H_0\): występowanie braków `y_t2` jest niezależne od analizowanej cechy skokowej;
- \(H_1\): występowanie braków `y_t2` zależy od analizowanej cechy skokowej.

Wartości p skorygowano metodą Holma.

In [4]:
categorical_results = []

for variable in ["trt", "x2"]:
    table = pd.crosstab(df[variable], df["R_y_t2"])

    chi2, p_value, dof, expected = stats.chi2_contingency(
        table,
        correction=False
    )

    phi = np.sqrt(chi2 / table.to_numpy().sum())

    print(f"Tabela kontyngencji: {variable} a wskaźnik braków")
    display(table.rename(columns={
        0: "y_t2 zaobserwowane (R=0)",
        1: "y_t2 brakujące (R=1)"
    }))

    missing_rate = (
        df.groupby(variable)["R_y_t2"]
        .mean()
        .mul(100)
        .rename("Odsetek braków y_t2 (%)")
    )
    display(missing_rate.to_frame().round(2))

    categorical_results.append({
        "Cecha": variable,
        "Chi-kwadrat": chi2,
        "Stopnie swobody": dof,
        "p-value": p_value,
        "Współczynnik phi": phi,
        "Minimalna liczebność oczekiwana": expected.min()
    })

categorical_results = pd.DataFrame(categorical_results)
categorical_results["p-value skorygowane (Holm)"] = multipletests(
    categorical_results["p-value"],
    method="holm"
)[1]

display(categorical_results.round(4))

Tabela kontyngencji: trt a wskaźnik braków


R_y_t2,y_t2 zaobserwowane (R=0),y_t2 brakujące (R=1)
trt,,
1,242,138
2,302,123


,Odsetek braków y_t2 (%)
trt,
1,36.32
2,28.94


Tabela kontyngencji: x2 a wskaźnik braków


R_y_t2,y_t2 zaobserwowane (R=0),y_t2 brakujące (R=1)
x2,,
0,286,132
1,258,129


,Odsetek braków y_t2 (%)
x2,
0,31.58
1,33.33


,Cecha,Chi-kwadrat,Stopnie swobody,p-value,Współczynnik phi,Minimalna liczebność oczekiwana,p-value skorygowane (Holm)
0,trt,4.9797,1,0.0256,0.0787,123.2050,0.0513
1,x2,0.2823,1,0.5952,0.0187,125.4745,0.5952


### Interpretacja wyników

W przypadku zmiennej `trt` wynik testu chi-kwadrat wskazuje na różnice w rozkładzie braków `y_t2` pomiędzy grupami. Siła tej zależności, mierzona współczynnikiem phi, jest niewielka.

Dla zmiennej `x2` nie stwierdzono istotnej zależności ze wskaźnikiem braków. Współczynnik phi wskazuje również na bardzo słaby związek.

Wyniki należy interpretować z uwzględnieniem korekty poziomu istotności dla wielokrotnych porównań.

## 4. Regresja logistyczna

W celu jednoczesnej oceny zależności występowania braków od wszystkich pozostałych cech oszacowano model regresji logistycznej. Zmienną objaśnianą jest wskaźnik `R_y_t2`.

Model ma postać:

$$
\operatorname{logit}\left(P(R_{y\_t2}=1)\right)
=
\beta_0+\beta_1 y\_t1+\beta_2 x1
+\beta_3 I(trt=2)+\beta_4 I(x2=1)
$$

Dla zmiennych `trt` i `x2` zastosowano kodowanie kategorii, w którym kategoriami referencyjnymi są odpowiednio `trt=1` i `x2=0`.

Wyniki przedstawiono za pomocą ilorazów szans (OR), 95-procentowych przedziałów ufności i wartości p. Model pozwala ocenić zależność każdej cechy od występowania braków przy jednoczesnym uwzględnieniu pozostałych zmiennych.

In [5]:
model = smf.logit(
    "R_y_t2 ~ y_t1 + x1 + C(trt) + C(x2)",
    data=df
).fit(disp=False, cov_type="HC3")

confidence_intervals = model.conf_int()

logit_results = pd.DataFrame({
    "Współczynnik": model.params,
    "OR": np.exp(model.params),
    "OR – 95% CI dolna granica": np.exp(confidence_intervals[0]),
    "OR – 95% CI górna granica": np.exp(confidence_intervals[1]),
    "p-value": model.pvalues
})

display(logit_results.round(4))
print(model.summary())

,Współczynnik,OR,OR – 95% CI dolna granica,OR – 95% CI górna granica,p-value
Intercept,-2.8505,0.0578,0.0068,0.4924,0.0091
C(trt)[T.2],-0.3780,0.6852,0.4809,0.9764,0.0364
C(x2)[T.1],-0.0639,0.9381,0.6792,1.2958,0.6984
y_t1,0.2103,1.2341,1.0600,1.4367,0.0067
x1,0.0008,1.0008,0.9744,1.0280,0.9508


                           Logit Regression Results                           
Dep. Variable:                 R_y_t2   No. Observations:                  805
Model:                          Logit   Df Residuals:                      800
Method:                           MLE   Df Model:                            4
Date:                Tue, 24 Nov 2026   Pseudo R-squ.:                 0.01232
Time:                        09:17:16   Log-Likelihood:                -500.91
converged:                       True   LL-Null:                       -507.16
Covariance Type:                  HC3   LLR p-value:                   0.01404
                  coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------
Intercept      -2.8505      1.093     -2.608      0.009      -4.992      -0.709
C(trt)[T.2]    -0.3780      0.181     -2.092      0.036      -0.732      -0.024
C(x2)[T.1]     -0.0639      0.165     -0.388    

In [6]:
# Porównanie modelu pełnego z modelem zawierającym wyłącznie wyraz wolny
null_model = smf.logit("R_y_t2 ~ 1", data=df).fit(disp=False)

lr_statistic = 2 * (model.llf - null_model.llf)
lr_df = int(model.df_model - null_model.df_model)
lr_p_value = stats.chi2.sf(lr_statistic, lr_df)

print(f"Statystyka testu ilorazu wiarygodności: {lr_statistic:.4f}")
print(f"Stopnie swobody: {lr_df}")
print(f"p-value: {lr_p_value:.4f}")
print(f"Pseudo-R² McFaddena: {model.prsquared:.4f}")

Statystyka testu ilorazu wiarygodności: 12.4927
Stopnie swobody: 4
p-value: 0.0140
Pseudo-R² McFaddena: 0.0123


### Interpretacja wyników modelu

Wyniki regresji logistycznej wskazują, że po jednoczesnym uwzględnieniu pozostałych cech istotne statystycznie są `y_t1` oraz `trt`. Wzrost `y_t1` wiąże się ze wzrostem szans wystąpienia braku `y_t2`. Szanse braku `y_t2` w grupie `trt=2` są niższe niż w grupie referencyjnej `trt=1`.

Dla `x1` i `x2` nie stwierdzono istotnego związku z występowaniem braków po uwzględnieniu pozostałych zmiennych.

Test ilorazu wiarygodności służy do oceny, czy model zawierający predyktory lepiej opisuje wskaźnik braków niż model z samym wyrazem wolnym.

## 5. Wnioski

W analizowanym zbiorze brakujące wartości występują w zmiennej `y_t2`. Przeprowadzone analizy wskazują, że prawdopodobieństwo braku tej zmiennej jest powiązane z niektórymi obserwowanymi cechami.

Najbardziej wyraźny związek stwierdzono dla `y_t1`. Zmienna ta różnicuje grupę obserwacji z brakującym i zaobserwowanym `y_t2`, a jej znaczenie potwierdza również model regresji logistycznej. Zmienna `trt` wykazuje zależność w analizie jednoczynnikowej i pozostaje istotna w modelu logistycznym, choć wynik testu chi-kwadrat po korekcie dla wielokrotnych porównań należy interpretować ostrożnie. Dla `x1` i `x2` nie uzyskano istotnych statystycznie wyników.

W związku z tym `y_t1` oraz `trt` warto uwzględnić jako zmienne pomocnicze podczas imputacji `y_t2`. Pozostałe cechy można również pozostawić w modelu imputacyjnym, jeżeli uzasadnia to ich znaczenie merytoryczne lub związek z imputowaną zmienną.

Uzyskane wyniki nie potwierdzają założenia MCAR, ponieważ występują zależności między wskaźnikiem braków a obserwowanymi cechami. Wyniki są zgodne z mechanizmem MAR, jednak na podstawie dostępnych danych nie można wykluczyć mechanizmu MNAR. Ostateczne rozróżnienie MAR i MNAR nie jest możliwe wyłącznie na podstawie obserwowanych wartości.

In [7]:
# Zestawienie najważniejszych wyników testów
summary_tests = pd.DataFrame([
    {
        "Zmienna": row["Cecha"],
        "Metoda": "Test t Welcha",
        "p-value": row["p-value"],
        "p-value skorygowane": row["p-value skorygowane (Holm)"]
    }
    for _, row in continuous_results.iterrows()
] + [
    {
        "Zmienna": row["Cecha"],
        "Metoda": "Test chi-kwadrat",
        "p-value": row["p-value"],
        "p-value skorygowane": row["p-value skorygowane (Holm)"]
    }
    for _, row in categorical_results.iterrows()
])

display(summary_tests.round(4))

,Zmienna,Metoda,p-value,p-value skorygowane
0,y_t1,Test t Welcha,0.0084,0.0168
1,x1,Test t Welcha,0.2936,0.2936
2,trt,Test chi-kwadrat,0.0256,0.0513
3,x2,Test chi-kwadrat,0.5952,0.5952
